# Fuzzy Factory Funnel Analysis

# Environment set-up

In [ ]:
import pandas as pd
import csv
import os
import glob
from sqlalchemy import create_engine
import time
import matplotlib.pyplot as plt
import numpy as np

# ETL Processing

## Stage 1: E - Extract

In [ ]:
folder_path = r"D:\Ecommerce\Dataset"

csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

for file_path in csv_files:
    file_name = os.path.basename(file_path)
    print(file_name)

In [ ]:
website_sessions = pd.read_csv(r'Dataset\website_sessions.csv')
products = pd.read_csv(r'Dataset\products.csv')
website_pageviews = pd.read_csv(r'Dataset\website_pageviews.csv')
orders = pd.read_csv(r'Dataset\orders.csv')
order_items = pd.read_csv(r'Dataset\order_items.csv')
order_item_refunds = pd.read_csv(r'Dataset\order_item_refunds.csv')

In [ ]:
table_list = ["website_sessions", "products", "website_pageviews",
              "orders", "order_items", "order_item_refunds"]

data_frames = {}

for table in table_list:
    print("=============================================================================================")
    print(f"Data {table}")
    
    file_path = os.path.join(folder_path, f"{table}.csv")
    if not os.path.exists(file_path):
        print(f"Not Contain {file_path}, Cancel.\n")
        continue
    
    try:
        df = pd.read_csv(file_path)
        
        if df.empty:
            print(f"File {table}.csv is Null, Cancel.\n")
            continue
        
        data_frames[table] = df
        print(f"First 5 row of {table}:")
        display(df.head())

        print("\nSummary:")
        df.info()

        print(f"\nNull Row Count:")
        print(df.isnull().sum())
        print(f"\nDulicate Row Count: {df.duplicated().sum()}")
    
    except Exception as e:
        print(f"Error {table}.csv: {e}\n")

## Stage 2: T - Transform

### Clean data

In [ ]:
# order_item_refunds
## Correct data format
order_item_refunds['created_at'] = pd.to_datetime(order_item_refunds['created_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

order_item_refunds.info()

In [ ]:
# order_items
## Correct data format
order_items['created_at'] = pd.to_datetime(order_items['created_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

order_items.info()

In [ ]:
# Orders
## Correct data type
orders['created_at'] = pd.to_datetime(orders['created_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

orders.info()


In [ ]:
# website_session
## Correct data type
website_sessions['created_at'] = pd.to_datetime(website_sessions['created_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

website_sessions.info()


In [ ]:
# website_pageviews
## Correct data type
website_pageviews['created_at'] = pd.to_datetime(website_pageviews['created_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
website_pageviews.info()

In [ ]:
# products
## Correct data format
products['created_at'] = pd.to_datetime(products['created_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
products.info()

### Check data validity

In [ ]:
table_list = ["website_sessions", "products", "website_pageviews",
              "orders", "order_items", "order_item_refunds"]

for table_name in table_list:
    df = globals().get(table_name)
    if df is None:
        print(f"Can't Find DataFrame: {table_name}")
        continue
    if not isinstance(df, pd.DataFrame):
        print(f"{table_name} is not DataFrame")
        continue

    print("=============================================================================")
    print(f"Data Summary {table_name}")
    print(df.describe(include='all'))

    numeric_cols = df.select_dtypes(include=['number']).columns
    if len(numeric_cols) == 0:
        print(f"{table_name} do not contain numerical values.\n")
        continue

    df_numeric = df[numeric_cols]

    axes = df_numeric.hist(bins=20, figsize=(20, 15), edgecolor='black', grid=False, density=True)

    plt.suptitle(f'Distribution - {table_name}', fontsize=20)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    print("\n")

No suspicious porblem in data validity

## Stage 3: L - Load

In [ ]:
dataframes = {
    'order_item_refunds': order_item_refunds,
    'website_sessions': website_sessions,
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'website_pageviews': website_pageviews,
}

db_user = '***'
db_password = '***'
db_host = '***'
db_port = '***'
db_name = '***'

engine = create_engine(f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')

print("Start loading data into PostgreSQL...")

for table_name, df in dataframes.items():
    start_time = time.time()
    print(f"Loading '{table_name}' with {len(df)} records...")
    df.to_sql(
        name=table_name, 
        con=engine, 
        if_exists='replace', 
        index=False, 
        chunksize=50000,  
        method='multi'    
    )
    
    end_time = time.time()
    print(f"Successful load '{table_name}' in {round(end_time - start_time, 2)} seconds.\n")

print("Done")